# Checkpoint 9: What is a baseline?

**Goal:** understand the simplest comparison before fitting a feature-based model.

A **baseline** is a deliberately simple prediction method. A more complicated model should provide a useful improvement over it. Here we distinguish:

1. **Always-negative classifier:** answers “no incident” for every example.
2. **Constant-probability baseline:** assigns every example the incident fraction observed in training.

These are different outputs: a yes/no answer versus a probability. The probability baseline does not inspect temperature or any other feature.

This notebook recomputes checkpoint 8's training subset from source JSON. It does not require another notebook or its CSV to exist. The 48 hour allowance and demonstration cutoff remain unchanged. No final test evaluation or feature-based model training occurs here.


## 1. Rebuild the reviewed training subset

The following setup repeats the previously reviewed helper definitions for independent execution. Focus on the baseline cells below; the temporal policy has not changed. Production extraction into shared Python modules remains a later checkpoint.


In [2]:
from pathlib import Path
from datetime import datetime, timedelta, timezone
from statistics import mean
import json
import math
import pandas as pd
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "data/events.jsonl").is_file()
             and (p / "src/dispatch_risk/contracts.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run from inside the candidate repository.")

def load(name):
    with (ROOT / "data" / name).open() as handle:
        return [json.loads(line) for line in handle if line.strip()]

raw_events = load("events.jsonl")
decisions = pd.DataFrame(load("decision_times.jsonl"))
reports = pd.DataFrame(load("labels.jsonl"))
decisions["decision_time"] = pd.to_datetime(decisions["decision_time"], utc=True)
for field in ("incident_at", "label_available_at"):
    reports[field] = pd.to_datetime(reports[field], utc=True)
HORIZON = pd.Timedelta(hours=6)
GRACE_HOURS = 48
WINDOW_HOURS = 3


In [3]:
def utc(text):
    value = datetime.fromisoformat(text.replace("Z", "+00:00"))
    if value.tzinfo is None:
        raise ValueError("An explicit timezone is required.")
    return value.astimezone(timezone.utc)

def known_revisions(records, checkpoint):
    """Select the highest revision received by the checkpoint for each event ID."""
    if checkpoint.tzinfo is None:
        raise ValueError("Checkpoint must include a timezone")
    checkpoint = checkpoint.astimezone(timezone.utc)
    delivered = {}
    latest = {}
    for event in records:
        if utc(event["received_at"]) > checkpoint:
            continue
        key = (event["event_id"], event["revision"])
        # Compare canonical timestamps so equivalent timezone notation agrees.
        normalized = dict(event)
        for field in ("device_time", "received_at"):
            normalized[field] = utc(event[field]).isoformat()
        signature = json.dumps(normalized, sort_keys=True, allow_nan=False)
        if key in delivered:
            if delivered[key] != signature:
                raise ValueError("Conflicting content for the same event/revision")
            continue
        delivered[key] = signature
        prior = latest.get(event["event_id"])
        if prior is not None and prior["shipment_id"] != event["shipment_id"]:
            raise ValueError("An event ID changed shipment")
        if prior is None or event["revision"] > prior["revision"]:
            latest[event["event_id"]] = dict(event)
    return [latest[event_id] for event_id in sorted(latest)]

def first_features(records, shipment, checkpoint):
    selected = known_revisions(records, checkpoint)
    usable = []
    for event in selected:
        if event["shipment_id"] != shipment or event["kind"] != "temperature_c":
            continue
        value = event["value"]
        if isinstance(value, bool) or not isinstance(value, (float, int)):
            continue
        if not math.isfinite(value):
            continue
        measured = utc(event["device_time"])
        received = utc(event["received_at"])
        if measured > received:  # Exclude measurements whose clocks run ahead of receipt.
            continue
        usable.append(event)
    if not usable:
        return {"latest_temperature_c": None, "measurement_age_minutes": None,
                "arrival_delay_minutes": None, "temperature_missing": 1}
    latest = max(usable, key=lambda e: (utc(e["device_time"]),
                                      utc(e["received_at"]), e["event_id"]))
    measured = utc(latest["device_time"])
    received = utc(latest["received_at"])
    return {
        "latest_temperature_c": float(latest["value"]),
        "measurement_age_minutes": (checkpoint - measured).total_seconds() / 60,
        "arrival_delay_minutes": (received - measured).total_seconds() / 60,
        "temperature_missing": 0,
    }

def window_features(records, shipment, checkpoint, window_hours=3):
    if checkpoint.tzinfo is None:
        raise ValueError("Checkpoint needs an explicit timezone")
    if not math.isfinite(window_hours) or window_hours <= 0:
        raise ValueError("Window must be finite and positive")
    left = checkpoint - timedelta(hours=window_hours)
    points = []
    for event in known_revisions(records, checkpoint):
        if event["shipment_id"] != shipment or event["kind"] != "temperature_c":
            continue
        value = event["value"]
        if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value):
            continue
        measured, received = utc(event["device_time"]), utc(event["received_at"])
        # Use the same clock rule as the latest-temperature feature.
        if measured > received or not left < measured <= checkpoint:
            continue
        points.append((measured, event["event_id"], float(value)))
    points.sort(key=lambda p: (p[0], p[1]))
    if not points:
        return {"temperature_count": 0, "temperature_mean_c": None,
                "temperature_max_c": None, "temperature_trend_c_per_hour": None,
                "temperature_span_hours": None}
    values = [p[2] for p in points]
    hours = [(p[0] - points[0][0]).total_seconds() / 3600 for p in points]
    x_mean, y_mean = mean(hours), mean(values)
    denominator = sum((x - x_mean)**2 for x in hours)
    slope = (sum((x - x_mean)*(y - y_mean) for x, y in zip(hours, values))
             / denominator) if denominator > 0 else None
    return {"temperature_count": len(points), "temperature_mean_c": y_mean,
            "temperature_max_c": max(values), "temperature_trend_c_per_hour": slope,
            "temperature_span_hours": hours[-1]}

def eligibility_table(checkpoints, incident_reports, training_cutoff, grace_hours=48):
    if training_cutoff.tzinfo is None:
        raise ValueError("Training cutoff must be timezone-aware")
    if grace_hours < 0:
        raise ValueError("Reporting allowance cannot be negative")
    grace = pd.Timedelta(hours=grace_hours)
    # Future report contents do not participate in historical label construction.
    known = incident_reports.loc[incident_reports["label_available_at"] <= training_cutoff]
    grouped = {sid: group for sid, group in known.groupby("shipment_id", sort=False)}
    result = []
    for row in checkpoints.itertuples(index=False):
        end = row.decision_time + HORIZON
        eligible_at = end + grace
        label = None
        if row.decision_time >= training_cutoff:
            status = "outside_training_period"
        elif eligible_at > training_cutoff:
            status = "waiting_for_maturity"
        else:
            group = grouped.get(row.shipment_id)
            positive = group is not None and bool(((group["incident_at"] > row.decision_time)
                                                   & (group["incident_at"] <= end)).any())
            label = int(positive)
            status = "eligible_positive" if positive else "eligible_assumed_negative"
        result.append({"shipment_id": row.shipment_id, "decision_time": row.decision_time,
                       "horizon_end": end, "eligible_at": eligible_at,
                       "training_cutoff": training_cutoff, "grace_hours": grace_hours,
                       "status": status, "label": label})
    output = pd.DataFrame(result)
    output["label"] = output["label"].astype("Int64")
    return output

In [4]:
unique_times = sorted(decisions["decision_time"].unique())
cutoff = pd.Timestamp(unique_times[min(int(len(unique_times)*0.70), len(unique_times)-1)])
all_statuses = eligibility_table(decisions, reports, cutoff, GRACE_HOURS)
eligible = all_statuses.loc[all_statuses["label"].notna()].copy()
events_by_shipment = {}
for event in raw_events:
    events_by_shipment.setdefault(event["shipment_id"], []).append(event)

FEATURE_COLUMNS = [
    "latest_temperature_c", "measurement_age_minutes", "arrival_delay_minutes",
    "temperature_missing", "temperature_count", "temperature_mean_c",
    "temperature_max_c", "temperature_trend_c_per_hour", "temperature_span_hours",
]
rows = []
for item in eligible.itertuples(index=False):
    checkpoint = item.decision_time.to_pydatetime()
    history = events_by_shipment.get(item.shipment_id, [])
    features = first_features(history, item.shipment_id, checkpoint)
    features.update(window_features(history, item.shipment_id, checkpoint, WINDOW_HOURS))
    rows.append({"shipment_id": item.shipment_id, "decision_time": item.decision_time,
                 **features, "label": int(item.label), "horizon_end": item.horizon_end,
                 "eligible_at": item.eligible_at, "training_cutoff": cutoff,
                 "grace_hours": GRACE_HOURS, "lookback_hours": WINDOW_HOURS})
training_table = pd.DataFrame(rows).sort_values(["decision_time", "shipment_id"]).reset_index(drop=True)
X = training_table[FEATURE_COLUMNS].copy()
y = training_table["label"].copy()
print("Training cutoff:", cutoff.isoformat())
print("Examples:", len(training_table), "Candidate feature columns:", len(FEATURE_COLUMNS))
print("Positive:", int(y.sum()), "Assumed negative:", int((y == 0).sum()))
display(training_table.head(8))


Training cutoff: 2026-02-22T23:00:00+00:00
Examples: 1209 Candidate feature columns: 9
Positive: 96 Assumed negative: 1113


,shipment_id,decision_time,latest_temperature_c,measurement_age_minutes,arrival_delay_minutes,temperature_missing,temperature_count,temperature_mean_c,temperature_max_c,temperature_trend_c_per_hour,temperature_span_hours,label,horizon_end,eligible_at,training_cutoff,grace_hours,lookback_hours
0,s-00000,2026-01-01 08:00:00+00:00,3.798,60.0,35.0,0,2,4.242500,4.687,-0.8890,1.0,0,2026-01-01 14:00:00+00:00,2026-01-03 14:00:00+00:00,2026-02-22 23:00:00+00:00,48,3
1,s-00000,2026-01-01 11:00:00+00:00,5.386,60.0,2.0,0,1,5.386000,5.386,NaN,0.0,1,2026-01-01 17:00:00+00:00,2026-01-03 17:00:00+00:00,2026-02-22 23:00:00+00:00,48,3
2,s-00001,2026-01-01 11:00:00+00:00,4.436,60.0,0.0,0,2,4.060000,4.436,0.7520,1.0,0,2026-01-01 17:00:00+00:00,2026-01-03 17:00:00+00:00,2026-02-22 23:00:00+00:00,48,3
3,s-00000,2026-01-01 14:00:00+00:00,9.857,0.0,0.0,0,3,8.977667,9.857,1.1125,2.0,1,2026-01-01 20:00:00+00:00,2026-01-03 20:00:00+00:00,2026-02-22 23:00:00+00:00,48,3
4,s-00001,2026-01-01 14:00:00+00:00,6.839,0.0,0.0,0,3,5.270000,6.839,1.0275,2.0,0,2026-01-01 20:00:00+00:00,2026-01-03 20:00:00+00:00,2026-02-22 23:00:00+00:00,48,3
5,s-00002,2026-01-01 14:00:00+00:00,4.184,60.0,2.0,0,2,4.517000,4.850,-0.6660,1.0,0,2026-01-01 20:00:00+00:00,2026-01-03 20:00:00+00:00,2026-02-22 23:00:00+00:00,48,3
6,s-00001,2026-01-01 17:00:00+00:00,7.380,60.0,2.0,0,1,7.380000,7.380,NaN,0.0,1,2026-01-01 23:00:00+00:00,2026-01-03 23:00:00+00:00,2026-02-22 23:00:00+00:00,48,3
7,s-00002,2026-01-01 17:00:00+00:00,4.325,60.0,2.0,0,2,4.406000,4.487,-0.1620,1.0,0,2026-01-01 23:00:00+00:00,2026-01-03 23:00:00+00:00,2026-02-22 23:00:00+00:00,48,3


## 2. Why accuracy alone can mislead

There are many more assumed-negative examples than positive examples in this subset. A method that always says “no incident” gets every negative correct, but misses every positive.

**Accuracy:** fraction of all yes/no answers that are correct.

**Recall:** fraction of actual incidents that the method catches.

High accuracy with zero recall is not useful incident detection. These are demonstrations on training data, not estimates of future performance. Labels continue to depend on our completeness assumption.


In [5]:
import numpy as np

labels_array = y.to_numpy(dtype=int)
negative_predictions = np.zeros(len(labels_array), dtype=int)
positive_count = int(labels_array.sum())
negative_count = len(labels_array) - positive_count
accuracy = float((negative_predictions == labels_array).mean())
recall = 0.0 if positive_count else float("nan")
print(f"Training examples: {len(labels_array):,}")
print(f"Positives: {positive_count}; assumed negatives: {negative_count}")
print(f"Always-negative training accuracy: {accuracy:.2%}")
print(f"Incident recall: {recall:.2%}")
assert accuracy == negative_count / len(labels_array)
if positive_count:
    assert recall == 0


Training examples: 1,209
Positives: 96; assumed negatives: 1113
Always-negative training accuracy: 92.06%
Incident recall: 0.00%


## 3. A constant probability baseline

Compute the incident fraction from training labels only: positives divided by all training examples. Return that same probability for every shipment checkpoint.

For this subset, the result is approximately 7.94%. This is a learned overall frequency, not a temperature-dependent prediction and not proof that every individual shipment has that risk.

**What has been learned?** One number from y. X is ignored. A later model will try to use X to distinguish higher-risk and lower-risk examples.

At prediction time the baseline does not need the new shipment's label. It simply returns the number previously learned from training. Do not recompute that number from validation/test outcomes before predicting them.


In [6]:
def fit_constant_probability(training_labels):
    values = np.asarray(training_labels, dtype=float)
    if values.size == 0 or not np.isin(values, [0, 1]).all():
        raise ValueError("Need nonempty binary training labels")
    return float(values.mean())

def constant_predictions(probability, count):
    if not 0 <= probability <= 1 or count < 0:
        raise ValueError("Invalid probability or count")
    return np.full(count, probability, dtype=float)

baseline_probability = fit_constant_probability(labels_array)
predicted_probabilities = constant_predictions(baseline_probability, len(labels_array))
print(f"Probability learned from training: {baseline_probability:.4%}")
print("Predictions for three new checkpoints:", constant_predictions(baseline_probability, 3))
assert np.all(predicted_probabilities == baseline_probability)
assert fit_constant_probability([0, 0, 0, 1]) == 0.25


Probability learned from training: 7.9404%
Predictions for three new checkpoints: [0.07940447 0.07940447 0.07940447]


## 4. How do we measure a probability error?

A yes/no accuracy calculation cannot fully describe probability quality. Start with the **Brier score**:

For each row, subtract the actual label from the predicted probability, square the difference, then average across rows. Lower is better; zero means every prediction exactly matches its binary outcome.

Example at predicted probability 0.20:
- If the outcome is 0: squared error is (0.20 − 0)² = 0.04.
- If the outcome is 1: squared error is (0.20 − 1)² = 0.64.

Do not interpret a single predicted 20% probability followed by no incident as proof that the probability was wrong. Probability quality is assessed over many outcomes. Brier score summarizes errors; it is not by itself a complete calibration diagnosis or launch criterion.


In [7]:
def brier_score(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    if actual.shape != predicted.shape or actual.size == 0:
        raise ValueError("Need aligned nonempty arrays")
    if not np.isin(actual, [0, 1]).all() or not np.isfinite(predicted).all():
        raise ValueError("Need binary outcomes and finite probabilities")
    if ((predicted < 0) | (predicted > 1)).any():
        raise ValueError("Probabilities must lie between zero and one")
    return float(np.mean((predicted - actual)**2))

assert abs(brier_score([0, 1], [0.2, 0.2]) - 0.34) < 1e-12
assert brier_score([0, 1], [0, 1]) == 0
comparison = pd.DataFrame([
    {"method": "Zero probability everywhere", "training_brier": brier_score(labels_array, negative_predictions)},
    {"method": "Training-frequency constant", "training_brier": brier_score(labels_array, predicted_probabilities)},
])
display(comparison)
print(comparison.to_string(index=False))
assert brier_score(labels_array, predicted_probabilities) <= brier_score(labels_array, negative_predictions)
print("This comparison uses training outcomes. It is not a held-out performance report.")


,method,training_brier
0,Zero probability everywhere,0.079404
1,Training-frequency constant,0.073099


                     method  training_brier
Zero probability everywhere        0.079404
Training-frequency constant        0.073099
This comparison uses training outcomes. It is not a held-out performance report.


## 5. What this experiment does and does not establish

We have a reproducible constant comparison model and an illustration of misleading accuracy. The training-frequency constant minimizes training squared probability error among constant predictions. Its lower training Brier score is therefore expected; it does not establish future performance or prove a useful model.

We still need temporal validation/test sets, input preparation settings fitted only on training data, and an evaluation that includes average precision, probability quality, and slices. A constant cannot rank individual shipments by risk because it gives them all the same score.

The current input features still contain missing values. The baseline ignores X, so no filling missing values is necessary here. Logistic regression will need an explicit missing-value policy and scaling, learned from training data only.

**Interview notes:** “I used the training incident fraction as a constant probability baseline. I checked incident recall as well as accuracy because always predicting negative looks accurate on an imbalanced dataset.”

**Try explaining this:** a method gets about 92% accuracy but catches zero incidents. Why would we not call it a successful risk engine?

**Next checkpoint:** explain logistic regression as learning weights for features, then a probability mapping; compare that idea with a decision tree. Establish temporal evaluation and preparing model inputs before fitting and comparing candidate models.
